# dev 일부 — InternVL 재확인 프롬프트 원인 점검

기존 결과는 읽기만 합니다. dev 40장(Q1/Q2 80문항)을 자산·정상/이상·Q1 보기 순서로 분산해 고정합니다. 모델 답으로 샘플을 고르지 않습니다.

1. **baseline**: 기존 기준선 프롬프트 그대로 새 추론 (현재 환경에서 답 재현 여부)
2. **review_only**: 기준선 + 중립적 재검토 지시, 이전 답 없음
3. **prior_only**: 기준선 + 그 문항 자신의 이전 답, 재검토 지시 없음
4. **current_agent**: 기존 agent reask 프롬프트 그대로

모든 조건은 같은 검색 정상 참조(diffdate)와 대상 이미지, 보기, greedy 추론을 씁니다. 확대·보기 정의·다른 문항의 답은 쓰지 않습니다.

**새 L4 런타임**, Secret `GITHUB_TOKEN` 노트북 액세스, 기존 Drive ZIP/dev/기준선 입력 필요. 320회 신규 호출(InternVL만), 대략 10~20분 예상이며 설치·Drive 복사 시간은 추가입니다. 실제 GPU 실행 시간은 아직 측정하지 않았습니다.

20문항마다 저장. 오류가 있으면 추론 셀이 중단되고 성공한 예측을 보존하며, 같은 셀 재실행으로 실패만 재시도합니다. 설정 선택·확정·GitHub 자동 게시 기능은 없습니다. raw 응답/전체 프롬프트/이미지 해시·크기·예상 InternVL patch 수/표/CI를 저장합니다.

출력: `MyDrive/lvlm_insplad_results/gold_v2_prompt_diagnostic_v1/`. 이전 `gold_v2_verify_agent_dev/`와 `frozen_config.json`은 건드리지 않습니다. 조건이나 입력을 바꾸면 `RUN_FOLDER`를 새 diagnostic 이름으로 바꾸세요.

current_agent는 baseline의 숨은 메타데이터 금지 문장도 빠져 있으므로 완전한 2×2 factorial은 아닙니다. 이 소규모 진단의 CI는 탐색용이고 gold 성능의 근거가 아닙니다.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_prompt_diagnostic_v1"
MODELS = ["internvl3_5_8b"]  # 필요하면 "qwen3_5_4b" 추가 (호출 2배)
N_IMAGES = 40
SEED = 42
CHECKPOINT_EVERY = 20
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DEV_REFS_DIR = DEV_DIR / "refs"
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"

REPO_DIR = Path("/content/lvlm_prompt_diagnostic_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
if not REPO_DIR.exists():
    run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError("VRAM 20GB 이상 L4/A100이 필요합니다.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"dev 세트가 FROZEN이 아닙니다: {summary.get('quality_status')}")
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_REFS_DIR / "retrieved_refs.csv"
need = [DEV_QA, REFS_CSV] + [STAGE1_DIR / m / "normal_ref_retrieved_diffdate.jsonl" for m in MODELS]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("기존 dev 기준선 입력 없음: " + ", ".join(missing))
print(f"입력 확인 완료 | {N_IMAGES}장, {N_IMAGES*2}문항, 신규 호출 {N_IMAGES*2*4*len(MODELS)}회")


In [ ]:
# 진단 runner 내장: GitHub push 없이 실행 가능
RUNNER_SOURCE = '#!/usr/bin/env python3\n"""Paired dev-only prompt diagnosis; never selects or freezes a gold configuration."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path[:0] = [str(ROOT / "scripts"), str(ROOT / "src")]\nimport run_gold_v2_verify_agent as agent\nimport run_gold_v2_dev_baselines as dev\n\nCONDITIONS = ("baseline", "review_only", "prior_only", "current_agent")\nVERSION = 1\nREVIEW = "Re-examine the TARGET carefully and answer this question independently."\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n\ndef build_prompt(row, own_first, condition):\n    baseline = dev.build_prompt(row, "normal_ref_retrieved_diffdate")\n    if condition == "baseline":\n        return baseline\n    if condition == "current_agent":\n        return agent.build_prompt(row, own_first, "reask")\n    if condition == "review_only":\n        extra = REVIEW\n    elif condition == "prior_only":\n        if own_first not in row["options"]:\n            raise ValueError("diagnosis requires a valid own stage-1 answer")\n        extra = f\'The previous answer to this question was "{own_first}. {row["options"][own_first]}".\'\n    else:\n        raise ValueError(condition)\n    return baseline.replace("\\n\\nQuestion:", f"\\n\\n{extra}\\n\\nQuestion:", 1)\n\n\ndef select_subset(rows, n_images=40, seed=42):\n    """Round-robin asset/GT/Q1 option-order strata, without using model outputs."""\n    if not 1 <= n_images <= 80:\n        raise ValueError("use 1..80 dev images; this runner is a small diagnosis")\n    by_core = defaultdict(dict)\n    for row in rows:\n        if row.get("protocol") != "gold_v2_dev":\n            raise ValueError("only gold_v2_dev questions are allowed")\n        if row["question_type"] not in (agent.Q1, agent.Q2):\n            continue\n        if row["question_type"] in by_core[row["core_id"]]:\n            raise ValueError("duplicate question in a core")\n        by_core[row["core_id"]][row["question_type"]] = row\n    strata = defaultdict(list)\n    for core, qs in by_core.items():\n        if set(qs) != {agent.Q1, agent.Q2} or qs[agent.Q1]["image"] != qs[agent.Q2]["image"]:\n            raise ValueError("each image needs its own Q1 and Q2")\n        q = qs[agent.Q1]\n        strata[(q["asset_category"], q["anomaly_gt"], tuple(q["options"].items()))].append(core)\n    if len(by_core) < n_images:\n        raise ValueError("not enough dev images")\n    for cores in strata.values():\n        cores.sort(key=lambda c: hashlib.sha256(f"{seed}:{c}".encode()).hexdigest())\n    assets = sorted({k[0] for k in strata})\n    keys = {a: sorted(k for k in strata if k[0] == a) for a in assets}\n    chosen = []\n    cursor = defaultdict(int)\n    while len(chosen) < n_images:\n        for asset in assets:\n            available = [k for k in keys[asset] if strata[k]]\n            if not available:\n                continue\n            key = available[cursor[asset] % len(available)]\n            cursor[asset] += 1\n            chosen.append(strata[key].pop(0))\n            if len(chosen) == n_images:\n                break\n    return [by_core[c][q] for c in chosen for q in (agent.Q1, agent.Q2)]\n\n\ndef main(argv=None, *, loader=None, infer=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in ("dev-qa", "stage1-dir", "refs-csv", "sup-zip", "unsup-zip", "out-dir"):\n        p.add_argument(f"--{name}", type=Path, required=True)\n    p.add_argument("--models", nargs="+", choices=list(dev.MODELS), default=["internvl3_5_8b"])\n    p.add_argument("--n-images", type=int, default=40)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--checkpoint-every", type=int, default=20)\n    p.add_argument("--n-boot", type=int, default=2000)\n    args = p.parse_args(argv)\n    # Protect previous results even if the caller gives the wrong output directory.\n    if not args.out_dir.name.startswith("gold_v2_prompt_diagnostic"):\n        raise ValueError("output must be a separate gold_v2_prompt_diagnostic* directory")\n    if len(args.models) != len(set(args.models)):\n        raise ValueError("duplicate models")\n    rows = select_subset(agent.read_jsonl(args.dev_qa), args.n_images, args.seed)\n    ids = {r["id"] for r in rows}\n    with args.refs_csv.open(encoding="utf-8-sig", newline="") as h:\n        refs = {r["image"]: r["retrieved_diffdate"] for r in csv.DictReader(h)}\n    first = {}\n    for model in args.models:\n        first[model] = agent.load_answers(args.stage1_dir / model / "normal_ref_retrieved_diffdate.jsonl")[0]\n        if any(first[model].get(r["id"]) not in r["options"] for r in rows):\n            raise ValueError("stage1 must cover every selected question with a valid choice")\n    if any(not refs.get(r["image"]) for r in rows):\n        raise ValueError("missing diffdate reference")\n    prompt_audit = [{"model_key": m, "condition": c, "id": r["id"],\n                     "own_first_choice": first[m][r["id"]], "prompt": build_prompt(r, first[m][r["id"]], c)}\n                    for m in args.models for c in CONDITIONS for r in rows]\n    identity = digest({"version": VERSION, "seed": args.seed, "rows": rows, "models": args.models,\n                       "refs": {r["image"]: refs[r["image"]] for r in rows}, "prompts": prompt_audit,\n                       "model_specs": {m: agent.base.MODEL_REGISTRY[m] for m in args.models}})\n    args.out_dir.mkdir(parents=True, exist_ok=True)\n    meta_path = args.out_dir / "run_metadata.json"\n    if meta_path.exists() and json.loads(meta_path.read_text())["identity"] != identity:\n        raise ValueError("output already contains different inputs; use a new diagnostic folder")\n    meta = {"protocol": "dev_prompt_diagnostic", "version": VERSION, "identity": identity,\n            "models": args.models, "images": args.n_images, "questions": len(rows), "seed": args.seed,\n            "model_specs": {m: agent.base.MODEL_REGISTRY[m] for m in args.models},\n            "decoding": "same infer_hf as baseline; InternVL greedy, do_sample=False",\n            "conditions": CONDITIONS, "benchmark_commit": agent.git_commit(), "gpu": agent.gpu_info(),\n            "selection": "diagnosis only; no selection.csv or frozen_config.json",\n            "sample": "round-robin asset / GT / Q1 option order; no model-output-based selection",\n            "caveat": "current_agent also omits the baseline hidden-metadata sentence; not a pure 2x2 factorial"}\n    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    agent.base.write_jsonl(args.out_dir / "subset_qa.jsonl", rows)\n    agent.base.write_jsonl(args.out_dir / "prompts.jsonl", prompt_audit)\n    targets = agent.base.prepare_images(rows, args.sup_zip, args.out_dir / "_images" / "target")\n    normals = {ref: agent.base.safe_extract(args.unsup_zip, ref, args.out_dir / "_images" / "reference")\n               for ref in sorted({refs[r["image"]] for r in rows})}\n    from PIL import Image\n    from insplad_benchmark.hf_adapters import _dynamic_preprocess\n    manifest = []\n    for member, path in list(targets.items()) + list(normals.items()):\n        with Image.open(path) as image:\n            manifest.append({"member": member, "path": str(path), "width": image.width, "height": image.height,\n                             "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n                             "internvl_patches_two_images": len(_dynamic_preprocess(image, max_num=3))})\n    agent.cons.write_csv(args.out_dir / "image_manifest.csv", manifest)\n    meta["image_identity"] = digest([{k: r[k] for k in ("member", "sha256")} for r in manifest])\n    # Per-prediction identity also binds actual extracted image bytes.\n    identity = digest([identity, meta["image_identity"]])\n    meta["prediction_identity"] = identity\n    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    if infer is None:\n        from insplad_benchmark.hf_adapters import infer_hf as infer\n    found, statuses, leaderboard, answers = {}, [], [], []\n    for model in args.models:\n        holder = agent.base.LazyHFAdapter(agent.base.MODEL_REGISTRY[model], loader=loader)\n        try:\n            for condition in CONDITIONS:\n                path = args.out_dir / "predictions" / model / f"{condition}.jsonl"\n                old = {r["id"]: r for r in (agent.read_jsonl(path) if path.exists() else [])}\n                done = []\n                called = reused = 0\n                for row in rows:\n                    qid = row["id"]\n                    record = old.get(qid)\n                    if record and record.get("identity") == identity and not record.get("error") and record.get("output", "").strip():\n                        reused += 1\n                    else:\n                        prompt = build_prompt(row, first[model][qid], condition)\n                        images = [normals[refs[row["image"]]], targets[row["image"]]]\n                        start = time.time()\n                        output, error = "", ""\n                        try:\n                            output = str(infer(holder.get(), images, prompt))\n                        except Exception as exc:\n                            error = f"{type(exc).__name__}: {exc}"\n                        record = {"id": qid, "identity": identity, "model_key": model, "condition": condition,\n                                  "own_first_choice": first[model][qid], "output": output, "error": error,\n                                  "predicted_choice": agent.base.parse_choice(output, set(row["options"])),\n                                  "latency_s": round(time.time() - start, 3), "images": list(map(str, images))}\n                        called += 1\n                    done.append(record)\n                    if args.checkpoint_every > 0 and len(done) % args.checkpoint_every == 0:\n                        agent.base.write_jsonl(path, done + [r for i, r in old.items() if i not in {d["id"] for d in done}])\n                        print(f"[{model}/{condition}] {len(done)}/{len(rows)} saved", flush=True)\n                agent.base.write_jsonl(path, done)\n                errors = sum(bool(r["error"]) for r in done)\n                invalid = sum(r["predicted_choice"] is None for r in done)\n                statuses.append({"model_key": model, "condition": condition, "questions": len(rows),\n                                 "called_this_run": called, "reused_predictions": reused, "row_errors": errors,\n                                 "invalid_answers": invalid, "status": "complete" if errors == 0 else "has_errors"})\n                final = {r["id"]: r["predicted_choice"] for r in done}\n                for r in rows:\n                    choice = final[r["id"]]\n                    answers.append({"model_key": model, "condition": condition, "id": r["id"],\n                                    "asset_category": r["asset_category"], "question_type": r["question_type"],\n                                    "answer": r["answer"], "stage1_choice": first[model][r["id"]],\n                                    "predicted_choice": choice, "correct": choice == r["answer"],\n                                    "predicted_text": r["options"].get(choice, "INVALID")})\n                if errors == 0:\n                    found[(model, condition)] = final\n                    leaderboard.append({"model_key": model, "condition": condition,\n                                        **agent.summary_row(rows, first[model], final, ids,\n                                                            {r["id"]: r["latency_s"] for r in done}),\n                                        "stage1_agreement": sum(final[i] == first[model][i] for i in ids) / len(ids)})\n        finally:\n            holder.release()\n    sig = agent.cons.significance_rows(agent.cons.image_pairs(rows), [], found, args.models,\n                                      [("baseline", c) for c in CONDITIONS[1:]], n_boot=args.n_boot)\n    sig = [r for r in sig if r["metric"] not in ("q3_accuracy", "q1q3_contradiction")]\n    for name, records in (("leaderboard.csv", leaderboard), ("answers.csv", answers), ("significance.csv", sig)):\n        agent.cons.write_csv(args.out_dir / name, records)\n    (args.out_dir / "run_status.json").write_text(json.dumps(statuses, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(statuses, indent=2), flush=True)\n    return 1 if any(s["row_errors"] for s in statuses) else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
RUNNER_PATH = REPO_DIR / "scripts/run_dev_prompt_diagnostic.py"
RUNNER_PATH.write_text(RUNNER_SOURCE, encoding="utf-8")


In [ ]:
# 중단되면 이 셀을 다시 실행: 성공한 호출은 재사용, 실패한 호출만 재시도
run([sys.executable, RUNNER_PATH, "--dev-qa", DEV_QA, "--stage1-dir", STAGE1_DIR,
     "--refs-csv", REFS_CSV, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--models", *MODELS, "--n-images", str(N_IMAGES),
     "--seed", str(SEED), "--checkpoint-every", str(CHECKPOINT_EVERY)])


In [ ]:
pd.set_option("display.max_rows", 200)
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status))
print("무효 답변 합계:", sum(s["invalid_answers"] for s in status))
display(pd.DataFrame(status))
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(lb[["model_key", "condition", "q1_anomaly_f1", "q2_defect_macro_f1",
            "q1_accuracy", "q2_accuracy", "joint_accuracy", "false_defect_rate", "stage1_agreement"]].round(3))
answers = pd.read_csv(RUN_ROOT / "answers.csv")
print("Q1 답 의미 분포")
display(pd.crosstab([answers.model_key, answers.condition], answers.predicted_text).reindex(columns=["Normal.", "Anomalous."], fill_value=0))
sig = pd.read_csv(RUN_ROOT / "significance.csv")
display(sig[~sig.metric.isin(["q3_accuracy", "q1q3_contradiction"])].round(3))
print("baseline의 stage1_agreement가 1보다 낮으면 실행 환경 차이도 함께 확인해야 합니다.")
print("current_agent는 기존 숨은 메타데이터 금지 문장도 빠집니다. 네 조건이 완전한 2×2 분리 실험은 아닙니다.")
print("결과 위치:", RUN_ROOT)


In [ ]:
# 결과 ZIP 다운로드 (Drive 결과는 그대로 유지)
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/dev_prompt_diagnostic_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in RUN_ROOT.rglob("*"):
        if p.is_file() and "_images" not in p.relative_to(RUN_ROOT).parts:
            z.write(p, p.relative_to(RUN_ROOT))
files.download(str(archive))
